In [ ]:
class Node:
    def __init__(self, y, feature_index, threshold_value):
        self.y = y
        self.feature_index = feature_index
        self.threshold_value = threshold_value
        self.prediction = self._get_node_prediction(self.y)
        self.left = None
        self.right = None
        
    def _get_node_prediction(self, y):
      values, counts = np.unique(y, return_counts=True)
      max_count_index = np.argmax(counts)
      pred = values[max_count_index]
      return pred

In [ ]:
class DecisionTreeClassifier():
    
    def __init__(self, max_depth = None):
        self.max_depth = max_depth
                
    def _convert_features(self, X, y=None):
      try:
            X = np.array(X)
      except:
          raise ValueError("A feature in X contains categorical values")

      if y is not None:
          try:
              y = np.array(y)
          except:
              raise ValueError("y contains categorical values")
      else:
          return X
      return X, y
  
    def _split(self, X, y, feature_index, threshold):
      x_left = X[X[:, feature_index] >= threshold]
      x_right = X[X[:, feature_index] < threshold]
      y_left = y[X[:, feature_index] >= threshold]
      y_right = y[X[:, feature_index] < threshold]

      return x_left, x_right, y_left, y_right
    
    def _calc_probability_weight(self, X, y, feature_index, threshold, split=True):
      if split:
          x_left, x_right, y_left, y_right = self._split(X, y, feature_index, threshold)

          p_left = np.unique(y_left, return_counts=True)[1] / y_left.shape[0]
          p_right = np.unique(y_right, return_counts=True)[1] / y_right.shape[0]

          w = (x_left.shape[0] / X.shape[0], x_right.shape[0] / X.shape[0])

          return (p_left, p_right), w
      else:
          p_left = np.unique(y, return_counts=True)[1] / y.shape[0]
          w = [1]
          return (p_left,), w
        
    def _calculate_entropy(self, X, y, feature_index, split=True):
      if len(np.unique(X[:, feature_index])) <= 2:
          entropy = 0
          threshold = 1
          p_both, w = self._calc_probability_weight(X, y, feature_index, threshold, split)

          for index, p in enumerate(p_both):
              entropy += np.sum(-(p * np.log2(p))) * w[index]

          return entropy, 1

      else:
          entropy = [None, None]
          feature_values = np.unique(X[:, feature_index])

          for value in feature_values:
              p_both, w = self._calc_probability_weight(X, y, feature_index, value, split)
              cur_entropy = 0
              for index, p in enumerate(p_both):
                  cur_entropy += (np.sum(-(p * np.log2(p))) * w[index])

              if entropy[0] is None or cur_entropy < entropy[0]:
                  entropy = [cur_entropy, value]

          return entropy
        
    def _calculate_information_gain(self, X, y, feature_index):
      entropy_before = self._calculate_entropy(X, y, 1, False)
      entropy_after = self._calculate_entropy(X, y, feature_index, True)
      information_gain = entropy_before[0] - entropy_after[0]
      return information_gain, entropy_after[1]
    
    def _select_best_feature(self, X, y, feature_indexes):
      info_gains = {}
      for index in feature_indexes:
          info_gains[index] = self._calculate_information_gain(X, y, index)

      info_gains = dict(sorted(info_gains.items(), key=lambda x: x[1][0], reverse=True))
      best_feature = next(iter(info_gains))
      feature_gain = info_gains[best_feature][0]
      feature_threshold = info_gains[best_feature][1]

      if feature_gain > 0:
          return best_feature, feature_gain, feature_threshold
      else:
          return None
        
    def _build_tree(self, X, y, depth=0):
      if self._select_best_feature(X, y, self.features):
          feature_idx, info_gain, threshold = self._select_best_feature(X, y, self.features)
          node = Node(y, feature_idx, threshold)

          if (depth < self.max_depth) and (len(np.unique(y)) > 1):
              x_left, x_right, y_left, y_right = self._split(X, y, feature_idx, threshold)
              node.left = self._build_tree(x_left, y_left, depth + 1)
              node.right = self._build_tree(x_right, y_right, depth + 1)
          return node
      else:
          return None
        
    def fit(self, X, y):
      self.X, self.y = self._convert_features(X, y)
      self.features = range(0, self.X.shape[1])

      if not self.max_depth:
          self.max_depth = self.X.shape[0]

      self.tree = self._build_tree(self.X, self.y)
      
    def _predict(self, X_test):
      tree = self.tree
      while tree:
          if X_test[tree.feature_index] >= tree.threshold_value and tree.left is not None:
              tree = tree.left
          elif X_test[tree.feature_index] < tree.threshold_value and tree.right is not None:
              tree = tree.right
          else:
              return tree.prediction
            
    def predict(self, X):
      X = self._convert_features(X)
      return [self._predict(x) for x in X]